# AutoFeatureFE — Titanic Demo

This notebook demonstrates the **autofeatureFE** agent loop on the Titanic dataset.

The agent iteratively improves a JSON feature-engineering pipeline using an LLM, keeping a **top-k pool** of the best pipelines found so far and penalising unnecessary feature bloat.

**Task**: binary classification — predict passenger survival  
**Metric**: AUC-ROC (higher = better)  
**Model**: XGBoost (fixed hyperparameters — only the features change)

## 1. Install dependencies

In [ ]:
%pip install -q xgboost scikit-learn pandas numpy matplotlib seaborn

## 2. Load & explore the Titanic dataset

In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json, subprocess, sys
from pathlib import Path

raw = sns.load_dataset('titanic')
print(f"Shape: {raw.shape}")
raw.head()

In [ ]:
# Missing values and dtypes
summary = pd.DataFrame({
    'dtype':   raw.dtypes,
    'missing': raw.isna().sum(),
    'missing%': (raw.isna().mean() * 100).round(1),
    'nunique': raw.nunique(),
})
summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

raw['survived'].value_counts().plot.bar(ax=axes[0], color=['#e74c3c','#2ecc71'])
axes[0].set_title('Survival (0=died, 1=survived)')
axes[0].set_xlabel('')

raw['age'].dropna().plot.hist(ax=axes[1], bins=30, edgecolor='white')
axes[1].set_title('Age distribution')

raw['fare'].dropna().plot.hist(ax=axes[2], bins=40, edgecolor='white')
axes[2].set_title('Fare distribution (right-skewed → good log target)')

plt.tight_layout()
plt.show()

## 3. Preprocess into numeric features

Our operation library works on **numeric** data.  
We do the minimal encoding here; the agent discovers richer combinations.

Features kept:
| Feature | Notes |
|---|---|
| `pclass` | 1 / 2 / 3 — already ordinal |
| `sex` | 0 = female, 1 = male |
| `age` | median-filled |
| `sibsp` | siblings / spouses aboard |
| `parch` | parents / children aboard |
| `fare` | ticket fare (right-skewed) |
| `embarked` | C=0, Q=1, S=2 |
| `age_missing` | flag for originally missing age |

The agent can discover: `family_size = sibsp + parch`, `fare_per_person = fare / family_size`, `log(fare)`, interactions between `pclass` and `fare`, etc.

In [ ]:
df = raw.copy()

# Target
y_col = 'survived'

# Flag missing age before filling
df['age_missing'] = df['age'].isna().astype(int)

# Fill missing values
df['age']      = df['age'].fillna(df['age'].median())
df['fare']     = df['fare'].fillna(df['fare'].median())
df['embarked'] = df['embarked'].fillna('S')

# Encode categoricals
df['sex']      = (df['sex'] == 'male').astype(int)
df['embarked'] = df['embarked'].map({'C': 0, 'Q': 1, 'S': 2})

# Select numeric features only
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'age_missing']
df_out = df[features + [y_col]].copy()

print(f"Final shape: {df_out.shape}")
print(f"Features: {features}")
df_out.head()

In [ ]:
# Save to CSV in the repo directory
REPO_DIR = Path('.')   # adjust if running from a different directory
csv_path = REPO_DIR / 'titanic.csv'
df_out.to_csv(csv_path, index=False)
print(f"Saved {len(df_out)} rows → {csv_path}")

## 4. Configure the system

In [ ]:
# Write task.json
task_cfg = {
    "task":          "classification",
    "dataset":       "csv",
    "csv_path":      "titanic.csv",
    "target_column": "survived",
    "metric":        "auc"
}
(REPO_DIR / 'task.json').write_text(json.dumps(task_cfg, indent=2))
print(json.dumps(task_cfg, indent=2))

In [ ]:
# Reset to baseline pipeline (standard scaling only)
baseline_pipeline = {
    "description": "baseline — no feature engineering, standard scaling",
    "steps": [
        {"op": "scale", "method": "standard"}
    ]
}
(REPO_DIR / 'pipeline.json').write_text(json.dumps(baseline_pipeline, indent=2))

# Clear previous results and pool so we start fresh
for f in ['results.tsv', 'topk_pool.json', 'data_profile_csv.md']:
    p = REPO_DIR / f
    if p.exists():
        p.unlink()
        print(f"Cleared {f}")

print("Ready — baseline pipeline set.")

## 5. Baseline evaluation

In [ ]:
import sys
sys.path.insert(0, str(REPO_DIR))

from prepare import prepare_data
X_train, X_val, y_train, y_val = prepare_data()
print(f"Train: {X_train.shape}  Val: {X_val.shape}")

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, RocCurveDisplay

baseline_model = XGBClassifier(
    n_estimators=1000, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    reg_lambda=1.0, random_state=42, n_jobs=-1,
    early_stopping_rounds=50, verbosity=0,
    objective='binary:logistic', eval_metric='auc',
)
baseline_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

baseline_auc = roc_auc_score(y_val, baseline_model.predict_proba(X_val)[:, 1])
print(f"Baseline AUC: {baseline_auc:.4f}")

fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_estimator(baseline_model, X_val, y_val, ax=ax, name='Baseline')
ax.set_title(f'Baseline ROC (AUC = {baseline_auc:.4f})')
plt.tight_layout()
plt.show()

## 6. Run the agent loop

Set your API key, then run the cell. The agent will run for `N_ITERATIONS` steps, each time:
1. Picking a base pipeline from the top-k pool
2. Asking the LLM to propose an improved pipeline
3. Evaluating it, updating the pool if it qualifies

> **To run longer / in the background**, use the terminal instead:
> ```bash
> ANTHROPIC_API_KEY=sk-... TOPK=5 uv run agent.py
> ```

In [ ]:
import os

# Set your key here (or export it in the shell before launching the notebook)
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'
# os.environ['OPENAI_API_KEY']    = 'sk-...'        # if using OpenAI
# os.environ['LLM_PROVIDER']      = 'openai'        # default: anthropic

N_ITERATIONS = 10   # increase for longer search

env = os.environ.copy()
env['TOPK']             = '5'
env['COMPLEXITY_ALPHA'] = '0.005'
env['INCLUDE_HISTORY']  = 'true'
env['HISTORY_SIZE']     = '20'
env['DATA_PROFILE']     = 'true'

print(f"Starting agent for {N_ITERATIONS} iterations...")
print("(progress appears below as iterations complete)\n")

for i in range(N_ITERATIONS):
    result = subprocess.run(
        [sys.executable, str(REPO_DIR / 'agent.py'), '--iterations', '1'],
        capture_output=True, text=True, env=env, cwd=str(REPO_DIR)
    )
    # Print last non-empty line as a progress indicator
    lines = [l for l in result.stdout.splitlines() if l.strip()]
    print(lines[-1] if lines else result.stderr.splitlines()[-1] if result.stderr else '(no output)')

> **Note**: if `agent.py` does not accept `--iterations`, run agent.py once per iteration is fine — it will pick up history from `results.tsv` automatically.

## 7. Results analysis

In [ ]:
results_path = REPO_DIR / 'results.tsv'

if not results_path.exists():
    print("No results.tsv yet — run the agent loop first (Section 6).")
else:
    results = pd.read_csv(results_path, sep='\t')
    print(f"{len(results)} experiments recorded")
    results.head(10)

In [ ]:
if results_path.exists():
    results = pd.read_csv(results_path, sep='\t')

    # Detect metric direction from results
    metric_col = 'val_score'
    metric_name = results['metric_name'].iloc[0] if 'metric_name' in results.columns else 'score'
    higher_is_better = metric_name == 'auc'

    if higher_is_better:
        running_best = results[metric_col].cummax()
    else:
        running_best = results[metric_col].cummin()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: metric per experiment + running best
    axes[0].scatter(results.index, results[metric_col], alpha=0.6, label='Each experiment', zorder=3)
    axes[0].plot(running_best, color='crimson', linewidth=2, label='Running best')
    axes[0].axhline(baseline_auc, linestyle='--', color='grey', label=f'Baseline ({baseline_auc:.4f})')
    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel(metric_name.upper())
    axes[0].set_title('AUC per iteration')
    axes[0].legend()

    # Right: n_features per experiment
    if 'n_features' in results.columns:
        axes[1].bar(results.index, results['n_features'], color='steelblue', alpha=0.7)
        axes[1].set_xlabel('Iteration')
        axes[1].set_ylabel('# Features')
        axes[1].set_title('Feature count per iteration')

    plt.suptitle('AutoFeatureFE — Titanic Agent Run', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# Summary table
if results_path.exists():
    results = pd.read_csv(results_path, sep='\t')
    best_idx = results[metric_col].idxmax() if higher_is_better else results[metric_col].idxmin()
    best = results.loc[best_idx]

    improvement = (best[metric_col] - baseline_auc) / baseline_auc * 100
    print(f"Baseline AUC : {baseline_auc:.4f}")
    print(f"Best AUC     : {best[metric_col]:.4f}  (iteration {best_idx})")
    print(f"Improvement  : {improvement:+.2f}%")
    if 'n_features' in results.columns:
        print(f"Best pipeline features: {int(best['n_features'])}")
    print()
    print("Top 5 experiments:")
    display(results.nlargest(5, metric_col) if higher_is_better else results.nsmallest(5, metric_col))

## 8. Inspect the best pipeline

In [ ]:
# Show the current top pool
pool_path = REPO_DIR / 'topk_pool.json'
if pool_path.exists():
    pool = json.loads(pool_path.read_text())
    print(f"Top-{len(pool)} pool:\n")
    for i, entry in enumerate(pool, 1):
        print(f"Rank {i}: score={entry['score']:.6f}  {entry['pipeline']['description']}")
        print(f"  Steps: {[s['op'] for s in entry['pipeline']['steps']]}\n")
else:
    print("No pool file yet — run the agent first.")

In [ ]:
# Show best pipeline in full
best_pipeline = json.loads((REPO_DIR / 'pipeline.json').read_text())
print(f"Best pipeline: {best_pipeline['description']}\n")
print(json.dumps(best_pipeline['steps'], indent=2))

## 9. Feature importance from best pipeline

In [ ]:
import importlib
import prepare as _prepare
importlib.reload(_prepare)   # pick up any changes

X_train_best, X_val_best, y_train_best, y_val_best = _prepare.prepare_data()
print(f"Best pipeline — Train: {X_train_best.shape}  Val: {X_val_best.shape}")

In [ ]:
best_model = XGBClassifier(
    n_estimators=1000, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    reg_lambda=1.0, random_state=42, n_jobs=-1,
    early_stopping_rounds=50, verbosity=0,
    objective='binary:logistic', eval_metric='auc',
)
best_model.fit(X_train_best, y_train_best, eval_set=[(X_val_best, y_val_best)], verbose=False)

best_auc = roc_auc_score(y_val_best, best_model.predict_proba(X_val_best)[:, 1])
print(f"Best pipeline AUC: {best_auc:.4f}  (baseline: {baseline_auc:.4f})")

In [ ]:
# Feature importance
importances = best_model.feature_importances_
n_feats = X_train_best.shape[1]

# Try to get feature names from the pipeline (approximate)
feat_labels = [f'f{i}' for i in range(n_feats)]

fi_df = pd.DataFrame({'feature': feat_labels, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=True).tail(20)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

fi_df.plot.barh(x='feature', y='importance', ax=axes[0], legend=False, color='steelblue')
axes[0].set_title('Feature Importance (top 20)')
axes[0].set_xlabel('Importance score')

RocCurveDisplay.from_estimator(baseline_model, X_val, y_val, ax=axes[1], name=f'Baseline (AUC={baseline_auc:.4f})')
RocCurveDisplay.from_estimator(best_model, X_val_best, y_val_best, ax=axes[1], name=f'Best pipeline (AUC={best_auc:.4f})', color='crimson')
axes[1].set_title('ROC Curve Comparison')

plt.suptitle('AutoFeatureFE — Titanic Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Next steps

- **Run longer**: `TOPK=5 uv run agent.py` in the terminal for an extended search
- **Try a different dataset**: update `task.json` with any CSV — set `dataset: csv`, `csv_path`, and `target_column`
- **Switch to regression**: set `task: regression`, `metric: rmse`, use `california_housing` or any regression CSV
- **Change model**: edit `train.py` — the feature pipeline logic is fully separate
- **Tune agent**: `COMPLEXITY_ALPHA`, `HISTORY_SIZE`, `DATA_PROFILE` env vars